# Fine-tuning a masked language model

## Load Dataset:

In [7]:
from datasets import load_dataset

raw_datasets = load_dataset("stanfordnlp/imdb")

## Load Tokenizer:

In [8]:
from transformers import AutoTokenizer

ckpt = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(ckpt)

## Load Model:

In [9]:
from transformers import AutoModelForMaskedLM
import torch

ckpt = "distilbert-base-uncased"

base_model = AutoModelForMaskedLM.from_pretrained(ckpt)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [10]:
from transformers import AutoModelForMaskedLM
import torch

finetuned_ckpt = "tankgauravgt/distilbert-uncased-imdb-finetuned"

finetuned_model = AutoModelForMaskedLM.from_pretrained(finetuned_ckpt)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [11]:
import torch

text = "This is a great [MASK]."

inputs = tokenizer(text, return_tensors="pt")
token_logits = base_model(**inputs).logits

# Find the location of [MASK] and extract its logits
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
mask_token_logits = token_logits[0, mask_token_index, :]

# Pick the [MASK] candidates with the highest logits
top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()

for token in top_5_tokens:
    print(f"'>>> {text.replace(tokenizer.mask_token, tokenizer.decode([token]))}'")

'>>> This is a great deal.'
'>>> This is a great success.'
'>>> This is a great adventure.'
'>>> This is a great idea.'
'>>> This is a great feat.'


In [12]:
import torch

text = "This is a great [MASK]."

inputs = tokenizer(text, return_tensors="pt")
token_logits = finetuned_model(**inputs).logits

# Find the location of [MASK] and extract its logits
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
mask_token_logits = token_logits[0, mask_token_index, :]

# Pick the [MASK] candidates with the highest logits
top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()

for token in top_5_tokens:
    print(f"'>>> {text.replace(tokenizer.mask_token, tokenizer.decode([token]))}'")

'>>> This is a great film.'
'>>> This is a great movie.'
'>>> This is a great show.'
'>>> This is a great comedy.'
'>>> This is a great idea.'
